In [ ]:
import csv
from pathlib import Path
import pandas as pd
import numpy as np

BASE = Path.cwd()                     # the "US Elections" folder
RAW  = BASE / "raw"
PLAN = BASE / "trends_pull_urls.csv"

STATES = [
 "Alabama","Alaska","Arizona","Arkansas","California","Colorado","Connecticut",
 "Delaware","District of Columbia","Florida","Georgia","Hawaii","Idaho","Illinois",
 "Indiana","Iowa","Kansas","Kentucky","Louisiana","Maine","Maryland","Massachusetts",
 "Michigan","Minnesota","Mississippi","Missouri","Montana","Nebraska","Nevada",
 "New Hampshire","New Jersey","New Mexico","New York","North Carolina","North Dakota",
 "Ohio","Oklahoma","Oregon","Pennsylvania","Rhode Island","South Carolina",
 "South Dakota","Tennessee","Texas","Utah","Vermont","Virginia","Washington",
 "West Virginia","Wisconsin","Wyoming"]

In [ ]:
def read_geomap(path):
    """One Trends 'Interest by subregion' CSV -> Series indexed by the 51 states."""
    lines = [l for l in Path(path).read_text(encoding="utf-8-sig").splitlines() if l.strip()]
    h = next((i for i, l in enumerate(lines)
              if l.split(",")[0].strip() in ("Region", "Country")), None)
    if h is None:
        raise ValueError(f"{Path(path).name}: no 'Region' header — Interest-over-time file?")

    rows = list(csv.reader(lines[h:]))
    header, body = rows[0], [r for r in rows[1:] if r and r[0].strip()]
    if len(header) - 1 > 1:
        raise ValueError(f"{Path(path).name}: compared breakdown (rows sum to 100%), "
                         "not a single-term export")

    def num(x):
        x = x.strip().replace("%", "")
        if x in ("", "N/A"):  return np.nan
        if x.startswith("<"): return 0.5          # Trends' "<1"
        try:    return float(x)
        except ValueError: return np.nan

    return pd.Series({r[0].strip(): num(r[1]) for r in body},
                     dtype=float).reindex(STATES)

In [ ]:
plan = pd.read_csv(PLAN)

recs = []
for r in plan.itertuples():
    s = read_geomap(RAW / r.save_as)
    recs.append(pd.DataFrame({"state": s.index, "year": r.year,
                              "category": r.category, "term_id": r.term_id,
                              "term": r.term, "rsv": s.values}))
long = pd.concat(recs, ignore_index=True)      # keep this — z-scoring is easier long

col_order = (plan.sort_values(["category", "order"])
                 .drop_duplicates("term_id").term_id.tolist())

wide = (long.pivot_table(index=["state", "year"], columns="term_id", values="rsv")
            .reindex(columns=col_order)
            .reset_index()
            .sort_values(["state", "year"]))
wide.columns.name = None

print(wide.shape)          # (204, 19)
wide.head()

In [ ]:
print(wide.groupby("year").size())                    # 51 each
print(wide.isna().sum()[lambda s: s > 0])             # where Google returned nothing